# Mission 4: Steerability

**Goal:** given a story opening and three required words, get the model to write a natural story that uses all three. Your baseline metric is the share of practice prompts whose story uses all three words (higher is better); fluency is also reported.

**Constraint:** the base model is fixed, so `steer` may not change its weights, and each `steer` call has a time limit (see section 3).”

**What you design:** a steering function, `steer(model, tokenizer, prompt, words)`.

**What you submit:** your steering code and your team benchmark (no model).

Edit only the cells marked **YOUR CODE**. Cells that start with `%%writefile` save your code into the `submission/` folder, which you submit on Gradescope. After editing one of these cells, rerun it.

### 1. Setup

Do the setup steps each time you open the notebook:
1. In Colab, choose **Runtime → Change runtime type → T4 GPU**.
2. **Upload `common.py`.** Open the file browser on the left and upload it. Colab deletes uploaded files when the session ends, so upload it again each session.
3. **Run the cell below.**

`common.py` contains the shared code that staff also use for grading. Do not edit it.

In [ ]:
!pip install -q transformers datasets

import os
assert os.path.exists("common.py"), "Put common.py in the same folder as this notebook."
import common

common.set_seed()
os.makedirs("submission/benchmark_data", exist_ok=True)
print("Device:", common.get_device())

### 2. Load the base model and practice prompts

In [ ]:
model, tokenizer = common.load_model()

practice = common.make_word_prompts(100)
print("Example prompt:", practice[0])


### 3. Your steering function

`steer` gets a story opening and a list of required words, and returns a story. Only the text `steer` returns is scored, so words placed in the prompt do not count. Staff run `steer` on a separate set of prompts that is not released.

**Rules:**
- Every word of the story you return must be generated by the model. You may change the prompt and how the model generates, but you may not write, insert, or edit story text yourself (for example, adding the required words to the model's output).
- The model's weights must stay unchanged (checked automatically). If you add PyTorch hooks to the model (for example, to change its activations), remove them before `steer` returns.

**Time limit:** each call to `steer` handles one prompt and must take 10 seconds or less (`common.M4_STEER_TIME_LIMIT_S`, checked automatically). Staff and other teams run your `steer` on many prompts. Generating several stories and returning the best one is fine, as long as the whole call stays within 10 seconds.

**Word rate** is the share of stories that use all three words, are at least 30 words long, and read fluently (higher is better). **Fluency** is the base model's perplexity on those stories (lower is better). Run `help(common.generate_story)` in a cell to see generation options.

In [ ]:
%%writefile submission/steer.py
import common


def steer(model, tokenizer, prompt: str, words: list[str]) -> str:
    """Return a story continuing `prompt` that uses every word in `words`."""
    # YOUR CODE HERE (currently lists the words before the prompt; replace with your method)
    steering_prompt = f"Write a story with: {', '.join(words)}. {prompt}"
    return common.generate_story(model, tokenizer, steering_prompt)

In [ ]:
steer = common.load_function("submission/steer.py", "steer")

result = common.evaluate_steering(steer, model, tokenizer, practice)
print(f"Word rate: {result['word_rate']:.0%}   Fluency (perplexity): {result['fluency_ppl']:.2f}")
print("Example:", practice[0]["words"], "→", result["stories"][0][:300])

### 4. Your team benchmark

In M4, every team uses the same base model, so your benchmark evaluates **other teams' `steer` functions**.

Requirements:
- `team_benchmark(steer_fn, model, tokenizer)` returns **one number, where higher is better**
- it calls `steer_fn(model, tokenizer, prompt, words)`, which returns a story, on prompts you design
- it returns the same number every time it runs (seed any randomness)
- it finishes within `common.BENCHMARK_TIME_LIMIT_S` seconds, including the time `steer_fn` takes, and does not change the model's weights
- it must stay within that limit even if `steer_fn` takes the full 10 seconds per call.
- any data it needs goes in `submission/benchmark_data/`

**Where to put your benchmark data.** Choose whichever is easier:
- **Few examples:** write them as a Python list directly in the cell below.
- **More examples, or data you generate with a script:** save a file (for example, `.json` or `.txt`) in `submission/benchmark_data/` and read it inside `team_benchmark`.

In [ ]:
%%writefile submission/benchmark.py
"""Team benchmark."""
import os
import common

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "benchmark_data")

# Option 1: a short list of examples written here
# EXAMPLES = [
#     'First example...',
#     'Second example...',
# ]

# Option 2: a file in submission/benchmark_data/
# import json
# with open(os.path.join(DATA_DIR, "my_data.json")) as f:
#     EXAMPLES = json.load(f)


def team_benchmark(steer_fn, model, tokenizer) -> float:
    """Evaluate the steering function `steer_fn` and return one number. Higher must mean better."""
    # YOUR CODE HERE
    raise NotImplementedError("Write your team benchmark.")

In [ ]:
team_benchmark = common.load_function("submission/benchmark.py", "team_benchmark")

print("Your steer function:")
common.check_team_benchmark(team_benchmark, model, tokenizer, steer_fn=steer)

### 5. Save your submission

Run this when you are ready to submit. Then upload the zip file to Gradescope.

Your `submission/` folder should contain:
- `steer.py`
- `benchmark.py` and `benchmark_data/`

In [ ]:
common.package_submission("mission4")